# LAB 26 — Дерева та алгоритми дерев: практичні задачі

**Модуль 3 · Python Advanced · Автор: Viktor Nikoriak**

---

## Що ми закріплюємо?

Цей lab-ноутбук містить три практичні задачі та п'ять питань для співбесіди рівня Middle/Senior.

| Секція | Тема | Рівень |
|--------|------|--------|
| Задача 1 | Побудова збалансованого BST з відсортованого масиву | Середній |
| Задача 2 | Пошук найнижчого спільного предка (LCA) | Складний |
| Задача 3 | Серіалізація та десеріалізація дерева | Складний |
| Питання 1–5 | Співбесіда: Senior Python Engineer | Senior |

---

In [ ]:
# ── Базова реалізація Node — використовується у всіх задачах ─────────────────
from __future__ import annotations
from collections import deque
from typing import Optional


class Node:
    """Вузол бінарного дерева."""
    def __init__(self, value: int):
        self.value = value
        self.left:  Optional[Node] = None
        self.right: Optional[Node] = None

    def __repr__(self):
        return f"Node({self.value})"


def inorder(node, result=None):
    """In-order обхід (Лів → Корінь → Прав). Повертає список значень."""
    if result is None: result = []
    if node:
        inorder(node.left,  result)
        result.append(node.value)
        inorder(node.right, result)
    return result


def height(node) -> int:
    """Висота дерева. O(n)."""
    if node is None: return -1
    return 1 + max(height(node.left), height(node.right))


def is_valid_bst(node, min_val=float('-inf'), max_val=float('inf')) -> bool:
    """Валідація BST через передачу глобального діапазону."""
    if node is None: return True
    if not (min_val < node.value < max_val): return False
    return (is_valid_bst(node.left,  min_val, node.value) and
            is_valid_bst(node.right, node.value, max_val))


def bst_insert(node, value) -> Node:
    """Вставка значення у BST. Повертає (можливо новий) корінь."""
    if node is None: return Node(value)
    if value < node.value: node.left  = bst_insert(node.left,  value)
    elif value > node.value: node.right = bst_insert(node.right, value)
    return node


print("Базові структури завантажено ✓")

---

## Задача 1 — Побудова збалансованого BST з відсортованого масиву (Рівень: Середній)

---

### Постановка задачі

Дано відсортований масив унікальних цілих чисел. Побудуйте **мінімально збалансоване BST** — дерево з мінімальною можливою висотою.

**Чому звичайна послідовна вставка не підходить?**
Якщо вставляти `[1, 2, 3, 4, 5]` по черзі, BST виродиться у правий список з висотою 4. Оптимальна висота — $\lfloor \log_2(5) \rfloor = 2$.

**Алгоритм:**
1. Знайдіть середній елемент масиву → він стає коренем
2. Ліва половина → рекурсивно будується ліве піддерево
3. Права половина → рекурсивно будується праве піддерево

**Складність:** $O(n)$ час, $O(\log n)$ пам'ять (глибина рекурсії)

**Очікувана поведінка:**
```
sorted_arr = [1, 2, 3, 4, 5, 6, 7]
# Побудоване дерево:
#         4
#        / \
#       2   6
#      / \ / \
#     1  3 5  7
# Висота = 2  (мінімально можлива)
```

---

In [ ]:
def build_balanced_bst(sorted_arr: list[int]) -> Optional[Node]:
    """
    Побудова мінімально збалансованого BST з відсортованого масиву.
    Складність: O(n) час, O(log n) пам'ять.

    TODO:
    1. Базовий випадок: якщо sorted_arr порожній → повернути None
    2. Знайдіть індекс середнього елемента: mid = len(sorted_arr) // 2
    3. Створіть корінь з sorted_arr[mid]
    4. Рекурсивно побудуйте ліве піддерево з sorted_arr[:mid]
    5. Рекурсивно побудуйте праве піддерево з sorted_arr[mid+1:]
    6. Поверніть корінь
    """
    pass  # TODO: реалізуйте


# ── Тести ────────────────────────────────────────────────────────────────────
def run_task1_tests():
    print("=== Задача 1: Збалансоване BST ===")
    print()

    # Тест 1: 7 елементів
    arr7 = [1, 2, 3, 4, 5, 6, 7]
    tree7 = build_balanced_bst(arr7)
    h7 = height(tree7)
    valid7 = is_valid_bst(tree7)
    sorted7 = inorder(tree7)
    print(f"  [{'✓' if h7 == 2 else '✗'}] n=7: висота={h7} (очікується 2)")
    print(f"  [{'✓' if valid7 else '✗'}] n=7: валідне BST={valid7}")
    print(f"  [{'✓' if sorted7 == arr7 else '✗'}] n=7: in-order={sorted7}")
    print()

    # Тест 2: 10000 відсортованих елементів (пряма вставка → вироджене)
    import math
    arr_large = list(range(1, 10001))

    # Вироджене (пряма вставка)
    degenerate = None
    for v in arr_large[:100]:   # лише 100 для швидкості
        degenerate = bst_insert(degenerate, v)
    h_degen = height(degenerate)

    # Збалансоване
    balanced = build_balanced_bst(arr_large)
    h_bal = height(balanced)
    expected_h = math.floor(math.log2(len(arr_large)))

    print(f"  n=10000:")
    print(f"    Пряма вставка (100 елем): висота={h_degen}  ← наближається до n")
    print(f"    Збалансоване:             висота={h_bal}  ← очікується ~{expected_h}")
    print(f"    Валідне BST: {is_valid_bst(balanced)}")
    print(f"    Відсортоване: {inorder(balanced) == arr_large}")


run_task1_tests()

### Рішення (розгорніть після спроби)

<details>
<summary>▶ Показати рішення</summary>

```python
def build_balanced_bst(sorted_arr: list[int]) -> Optional[Node]:
    if not sorted_arr:
        return None
    mid = len(sorted_arr) // 2
    root = Node(sorted_arr[mid])
    root.left  = build_balanced_bst(sorted_arr[:mid])
    root.right = build_balanced_bst(sorted_arr[mid + 1:])
    return root
```

**Чому це збалансоване?** Середній елемент завжди ділить масив навпіл. Кожне рекурсивне дерево будується з масиву вдвічі меншого → висота = $\lfloor \log_2 n \rfloor$.

**Часова складність:** $T(n) = 2 \cdot T(n/2) + O(1)$ → $O(n)$ (Master Theorem, Case 1).

</details>

---

## Задача 2 — Найнижчий Спільний Предок (LCA) (Рівень: Складний)

---

### Постановка задачі

Знайдіть **Найнижчий Спільний Предок (Lowest Common Ancestor, LCA)** двох вузлів у бінарному дереві.

LCA — це найглибший вузол, який є предком обох цільових вузлів одночасно.

**Реальне застосування:** CSS-специфічність у DOM-дереві. Щоб знайти, які CSS-правила застосовуються до двох елементів, браузер шукає їх спільного батька у DOM-дереві.

```
        4
       / \
      2   6
     / \ / \
    1  3 5  7

LCA(1, 3) = 2  (спільний батько 1 і 3)
LCA(1, 7) = 4  (корінь — перший спільний предок)
LCA(5, 7) = 6
LCA(2, 6) = 4
```

**Алгоритм (Post-order DFS «бульбашка знизу вгору»):**
1. Базовий випадок: якщо вузол — `None`, або вузол == `node_a`, або вузол == `node_b` → повернути вузол
2. Рекурсивно шукаємо у лівому та правому піддереві
3. Якщо **обидва** поверли не-None → поточний вузол є LCA → повернути його
4. Інакше → повернути той результат, що не None (або None)

**Складність:** $O(n)$ час, $O(h)$ пам'ять

---

In [ ]:
def find_lca(root: Optional[Node], node_a: Node, node_b: Node) -> Optional[Node]:
    """
    Пошук Найнижчого Спільного Предка (LCA) у бінарному дереві.
    Використовує Post-order DFS: «бульбашка» результатів знизу вгору.

    TODO:
    1. Базовий випадок: якщо root is None → return None
    2. Якщо root == node_a або root == node_b → return root (знайшли цільовий вузол)
    3. Рекурсивно шукаємо зліва: left = find_lca(root.left, node_a, node_b)
    4. Рекурсивно шукаємо справа: right = find_lca(root.right, node_a, node_b)
    5. Якщо left і right обидва не None → root є LCA → return root
    6. Інакше → return left if left else right
    """
    pass  # TODO: реалізуйте


# ── Тести ────────────────────────────────────────────────────────────────────
def run_task2_tests():
    # Будуємо дерево:
    #       4
    #      / \
    #     2   6
    #    / \ / \
    #   1  3 5  7
    r = Node(4)
    r.left = Node(2);  r.right = Node(6)
    r.left.left  = Node(1);  r.left.right  = Node(3)
    r.right.left = Node(5);  r.right.right = Node(7)

    n1 = r.left.left    # Node(1)
    n2 = r.left         # Node(2)
    n3 = r.left.right   # Node(3)
    n5 = r.right.left   # Node(5)
    n6 = r.right        # Node(6)
    n7 = r.right.right  # Node(7)

    print("=== Задача 2: Найнижчий Спільний Предок (LCA) ===")
    print()

    cases = [
        (n1, n3, n2, "LCA(1,3)=2"),
        (n1, n7, r,  "LCA(1,7)=4"),
        (n5, n7, n6, "LCA(5,7)=6"),
        (n2, n6, r,  "LCA(2,6)=4"),
        (n1, n2, n2, "LCA(1,2)=2 (один є предком іншого)"),
    ]

    for a, b, expected, desc in cases:
        result = find_lca(r, a, b)
        ok = result is expected
        print(f"  [{'✓' if ok else '✗'}] {desc}: результат={result}")


run_task2_tests()

### Рішення (розгорніть після спроби)

<details>
<summary>▶ Показати рішення</summary>

```python
def find_lca(root, node_a, node_b):
    if root is None:
        return None
    if root is node_a or root is node_b:
        return root                  # знайшли один з цільових вузлів
    
    left  = find_lca(root.left,  node_a, node_b)
    right = find_lca(root.right, node_a, node_b)
    
    if left and right:
        return root                  # обидва вузли знайдені в різних піддеревах
    return left if left else right   # передаємо знайдений вузол нагору
```

**Типова помилка:** Спроба знайти повні шляхи від кореня до обох вузлів, зберегти їх у списки і потім порівняти. Це коштує $O(n)$ зайвої пам'яті і потребує двох проходів.

**Архітектурне рішення:** Post-order («знизу вгору») дозволяє виявити LCA за **один прохід** $O(n)$ і лише $O(h)$ пам'яті для стека рекурсії.

</details>

---

## Задача 3 — Серіалізація та Десеріалізація дерева (Рівень: Складний)

---

### Постановка задачі

Реалізуйте серіалізацію (дерево → рядок) та десеріалізацію (рядок → дерево) бінарного дерева.

**Навіщо?** Це реальне завдання баз даних та розподілених систем: збереження структури дерева у файл або передача через мережу.

**Алгоритм (Pre-order + спеціальний маркер для None):**
- `serialize`: Pre-order обхід. Якщо вузол — `None`, записуємо `'#'`. Значення розділяємо комою.
- `deserialize`: Читаємо значення по черзі. Якщо `'#'` — повертаємо `None`. Інакше — створюємо вузол і рекурсивно будуємо ліве та праве піддерево.

```
Дерево:           Серіалізація (Pre-order):
    4             "4,2,1,#,#,3,#,#,6,5,#,#,7,#,#"
   / \                  ↑
  2   6           Порядок: Корінь→Лів→Прав
 / \ / \          '#' = None (порожній вузол)
1  3 5  7
```

**Складність:** $O(n)$ для обох операцій.

---

In [ ]:
def serialize(root: Optional[Node]) -> str:
    """
    Серіалізація: дерево → рядок через Pre-order обхід.
    None кодується як '#'. Значення розділяються комою.

    TODO:
    1. Якщо root is None → return '#'
    2. Рекурсивно серіалізуємо: str(root.value) + ',' + serialize(left) + ',' + serialize(right)
    """
    pass  # TODO: реалізуйте


def deserialize(data: str) -> Optional[Node]:
    """
    Десеріалізація: рядок → дерево.

    TODO:
    1. Розбийте рядок за комою: tokens = iter(data.split(','))
    2. Напишіть внутрішню рекурсивну функцію _build(tokens):
       a. val = next(tokens)
       b. Якщо val == '#' → return None
       c. Створіть node = Node(int(val))
       d. node.left  = _build(tokens)
       e. node.right = _build(tokens)
       f. return node
    3. Викличте _build(tokens) і поверніть результат

    Підказка: використовуйте iter() для ефективного споживання токенів
    """
    pass  # TODO: реалізуйте


# ── Тести ────────────────────────────────────────────────────────────────────
def run_task3_tests():
    # Будуємо оригінальне дерево
    original = Node(4)
    original.left = Node(2);  original.right = Node(6)
    original.left.left   = Node(1);  original.left.right  = Node(3)
    original.right.left  = Node(5);  original.right.right = Node(7)

    print("=== Задача 3: Серіалізація та Десеріалізація ===")
    print()

    # Тест 1: Серіалізація
    serialized = serialize(original)
    print(f"  Серіалізовано: {serialized}")
    expected_serial = "4,2,1,#,#,3,#,#,6,5,#,#,7,#,#"
    print(f"  [{'✓' if serialized == expected_serial else '✗'}] Серіалізація (очікується: {expected_serial})")
    print()

    # Тест 2: Десеріалізація
    restored = deserialize(serialized)
    original_inorder  = inorder(original)
    restored_inorder  = inorder(restored)
    print(f"  In-order оригіналу: {original_inorder}")
    print(f"  In-order відновленого: {restored_inorder}")
    print(f"  [{'✓' if original_inorder == restored_inorder else '✗'}] Дерева однакові (in-order)")
    print(f"  [{'✓' if height(original) == height(restored) else '✗'}] Висота однакова: {height(original)} == {height(restored)}")
    print()

    # Тест 3: Цикл serialize → deserialize
    import random
    random.seed(42)
    random_tree = None
    vals = random.sample(range(1, 101), 15)
    for v in vals:
        random_tree = bst_insert(random_tree, v)
    cycle = deserialize(serialize(random_tree))
    ok = inorder(random_tree) == inorder(cycle)
    print(f"  [{'✓' if ok else '✗'}] Випадкове BST з 15 вузлів: serialize → deserialize → однакове")


run_task3_tests()

### Рішення (розгорніть після спроби)

<details>
<summary>▶ Показати рішення</summary>

```python
def serialize(root):
    if root is None:
        return '#'
    return f"{root.value},{serialize(root.left)},{serialize(root.right)}"


def deserialize(data: str):
    tokens = iter(data.split(','))  # iter() → O(1) next() без зсуву пам'яті
    
    def _build(tokens):
        val = next(tokens)
        if val == '#':
            return None
        node = Node(int(val))
        node.left  = _build(tokens)
        node.right = _build(tokens)
        return node
    
    return _build(tokens)
```

**Ключові архітектурні рішення:**
- `iter()` на рядку токенів → кожен `next()` споживає один токен за $O(1)$
- Pre-order гарантує, що перший токен при десеріалізації завжди є коренем
- `'#'` явно кодує `None` → відновлює точну структуру дерева (включно з порожніми гілками)

</details>

---

## Підсумок Lab (практична частина)

| Задача | Алгоритм | Складність |
|--------|----------|------------|
| Збалансоване BST | Divide & Conquer, рекурсія | $O(n)$ час, $O(\log n)$ пам'ять |
| LCA | Post-order DFS, «бульбашка» | $O(n)$ час, $O(h)$ пам'ять |
| Серіалізація | Pre-order DFS + None-маркер | $O(n)$ для обох |

---

---

# Питання для співбесіди — рівень Middle/Senior

---

## Питання 1: Складність та «Вироджені» дерева

> **Запитання:** Яка часова складність пошуку та вставки у BST? Опишіть крайній випадок, коли продуктивність повністю деградує.

**Коротка відповідь:**
Для **збалансованого** BST — $O(\log n)$. Але якщо вставляти вже відсортовані дані, дерево вироджується у правий зв'язний список, і складність колапсує до $O(n)$.

**Глибоке пояснення:**
Ефективність BST базується на принципі «поділяй і володарюй» — кожен крок відкидає половину даних. Це гарантується **лише якщо дерево збалансоване** (висота $h \approx \log_2 n$). Якщо вставляти дані `1, 2, 3, 4, 5`, кожен новий вузол іде виключно вправо → висота $h = n$ → пошук перетворюється на лінійне сканування. Справжня складність дерева — $O(h)$, а не $O(\log n)$.

**Рішення:**
```python
# Замість послідовної вставки — будуємо з відсортованого масиву:
def build_balanced_bst(sorted_ids):
    if not sorted_ids: return None
    mid = len(sorted_ids) // 2
    root = Node(sorted_ids[mid])  # середній елемент — завжди корінь
    root.left  = build_balanced_bst(sorted_ids[:mid])
    root.right = build_balanced_bst(sorted_ids[mid+1:])
    return root
```

**Типова помилка:** Кандидати кажуть «BST — це $O(\log n)$», не усвідомлюючи, що це справедливо лише для збалансованих дерев.

---

## Питання 2: Рекурсія проти Ітерації — стек системи vs явний стек

> **Запитання:** Чому рекурсивний DFS у production Python може зламати додаток? Як архітектурно переписати обхід, щоб уникнути цього?

**Коротка відповідь:**
Python має ліміт глибини рекурсії (~1000 кадрів). Глибоке вироджене дерево викликає `RecursionError`. Рішення — ітеративний DFS з явним `list` як стеком.

**Глибоке пояснення:**
Рекурсія неявно використовує Call Stack операційної системи. При кожному виклику функції «заморожується» стан поточного вузла й кладеться у стек. При глибині > 1000 системний стек переповнюється. Ітеративна версія переміщує цю роботу на heap (звичайна пам'ять Python), яка може обробляти мільйони елементів.

**Ітеративний Pre-order DFS:**
```python
def preorder_iterative(root):
    if not root: return []
    stack, result = [root], []
    while stack:
        node = stack.pop()         # LIFO
        result.append(node.value)
        if node.right: stack.append(node.right)  # правий першим
        if node.left:  stack.append(node.left)   # лівий на вершині стека
    return result
```

**Типова помилка:** При Pre-order класти лівого нащадка у стек першим. Через LIFO це спричинить обробку правого першим — повний хаос.

---

## Питання 3: DFS vs BFS — коли що обрати?

> **Запитання:** Для пошуку найкоротшого зв'язку між двома людьми у соціальній мережі vs безпечного видалення вкладеної структури папок — який обхід обрати?

**Коротка відповідь:**
**BFS** для найкоротшого шляху у соціальній мережі. **DFS Post-order** для видалення папок.

**Глибоке пояснення:**
BFS обходить рівень за рівнем — перший знайдений шлях **гарантовано є найкоротшим**. Під капотом — **Черга (FIFO)**. DFS Post-order спочатку обробляє всіх нащадків, потім батька — ідеально для видалення (не можна видалити папку до видалення файлів усередині). Під капотом — **Стек (LIFO)**.

**Python BFS — критична деталь:**
```python
from collections import deque
queue = deque([root])
node = queue.popleft()  # O(1) — завжди deque!
# queue.pop(0) на list — O(n) → весь алгоритм O(n²)!
```

**Типова помилка:** Використовувати `list.pop(0)` для BFS-черги. $O(n)$ зсув пам'яті перетворює $O(n)$ алгоритм на $O(n^2)$.

---

## Питання 4: Глобальний інваріант BST

> **Запитання:** Напишіть функцію перевірки BST. Чому перевірки лише `left < root < right` недостатньо?

**Коротка відповідь:**
Локальна перевірка не виявляє порушення глобального інваріанту. Вузол у правому піддереві може бути меншим за корінь — і локальна перевірка цього не побачить.

**Глибоке пояснення:**
```
       10
      /  \
     5    15
    / \
   3   12    ← 12 > 10, але знаходиться в лівому піддереві 10!
              Локальна перевірка: 12 > 5 ✓  — хибно-позитивний результат!
```

**Правильне рішення — передача меж:**
```python
def is_valid_bst(node, low=float('-inf'), high=float('inf')):
    if not node: return True
    if not (low < node.value < high): return False
    return (is_valid_bst(node.left,  low, node.value) and
            is_valid_bst(node.right, node.value, high))
```

**Типова помилка:** `return is_valid_bst(node.left) and is_valid_bst(node.right)` без передачі меж — вузол на 10 рівнів нижче може порушити обмеження кореня.

---

## Питання 5: Пам'ять та Pre-order vs In-order для відновлення дерева

> **Запитання:** Якщо вам дати тільки In-order масив BST — чи можна відновити точну структуру дерева? А з Pre-order масиву?

**Коротка відповідь:**
З **In-order** одного — **не можна**: безліч різних дерев дають однаковий відсортований масив. З **Pre-order** — **можна**: перший елемент завжди є коренем, а властивість BST дозволяє поділити рештку на ліве і праве піддерева.

**Глибоке пояснення:**
In-order BST = відсортований масив `[1, 2, 3, 4, 5]`. Але корінь може бути будь-яким елементом — ця інформація загублена. Pre-order `[3, 1, 2, 5, 4]` → перший елемент `3` є коренем. Значення `< 3` (`[1, 2]`) → ліве піддерево. Значення `> 3` (`[5, 4]`) → праве. Рекурсивно відновлюємо за $O(n \log n)$.

**Рекомендоване рішення для серіалізації:** Pre-order з `'#'` маркером для None (як у Задачі 3) — відновлює точну структуру за $O(n)$.

**Типова помилка:** Вважати, що in-order BST однозначно визначає структуру. Ні: і `[2, 1, 3]` (pre-order), і `[1, 2, 3]` (pre-order) дають in-order `[1, 2, 3]`.

---

## Зведена таблиця

| Концепція | Правило |
|-----------|--------|
| Складність BST | $O(h)$ → $O(\log n)$ збалансоване, $O(n)$ вироджене |
| DFS структура | Стек (LIFO) — рекурсивний або явний `list` |
| BFS структура | Черга (FIFO) — виключно `collections.deque` |
| In-order BST | Завжди відсортований результат |
| Валідація BST | Передавати `[min, max]` при рекурсії, не лише локальна перевірка |
| `deque.popleft()` | $O(1)$; `list.pop(0)` — $O(n)$; ніколи не плутати! |